# RIR Lens — Model Notebook

**Prototype only · synthetic generated data.** The proposal calls for smartphone video and a participant study. Neither is represented by these generated observations. The notebook demonstrates the intended ML workflow and a working model/API contract. Run from the repository root.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, SVG
from src.pipeline import generate, split_participants, make_examples, train_branch, metrics

df = generate()
print(f"{len(df)} repetitions | {df.set_id.nunique()} sets | {df.participant.nunique()} simulated participants")
display(df.head())


2105 repetitions | 192 sets | 24 simulated participants


## 1. Data visualization and engineering

Each row represents a completed repetition. The generated label is `RIR = total endpoint repetitions − current repetition`. Endpoint set length is used **only to construct labels**, not as a model input. The generator simulates two exercises, load variation, person-specific movement, velocity decline, and increasing oscillation-like features. The features are illustrative, **not measured physiological tremor**.

![RIR distribution](assets/rir_distribution.svg)


In [ ]:
display(df[["rir","velocity","velocity_loss","tremor_amplitude","tremor_band_power","load_fraction"]].describe().round(3))
print("Feature correlations with RIR (synthetic):")
display(df[["rir","velocity","velocity_loss","tremor_amplitude","tremor_band_power","rep"]].corr()[["rir"]].round(3))
display(SVG(filename="assets/feature_means.svg"))


### Causal repetition windows and participant split

Each example contains the current and up to three prior completed repetitions. Missing early history is zero-filled and marked by a mask. The feature vector includes exercise, load fraction, and the current repetition index. It never includes future frames, total set length, or endpoint status. The two model variants use matched repetitions.


In [ ]:
splits = split_participants(df)
X_velocity, y, groups, keys = make_examples(df, combined=False)
X_combined, _, _, _ = make_examples(df, combined=True)
print("Feature shapes:", X_velocity.shape, X_combined.shape)
print("Participants by split:", {k:len(v) for k,v in splits.items()})
assert not (set(splits["train"]) & set(splits["test"]))
assert not (set(splits["validation"]) & set(splits["test"]))
print("First example's four-step padding is masked; all values come from current or earlier reps.")


Feature shapes: (2105, 15) (2105, 23)
Participants by split: {'train': 16, 'validation': 4, 'test': 4}
First example's four-step padding is masked; all values come from current or earlier reps.


## 2. Model architecture and optimization

| Stage | Operation |
|---|---|
| Input | Flattened 4-repetition history, validity mask, exercise indicator, load fraction, rep index |
| Hidden layer 1 | Dense 32, ReLU |
| Hidden layer 2 | Dense 16, ReLU |
| Output | Dense 1, linear numerical RIR estimate |
| Training | Mean squared error, Adam (`lr=0.003`), batches up to 128, validation early stopping (patience 45) |

Two models share this architecture. The velocity branch uses mean velocity and velocity loss; the combined branch adds candidate tremor amplitude and band power. Normalization parameters are fitted on training participants only. This small MLP is a demonstration architecture; the proposal's causal TCN remains a later experiment with real video-derived data.


In [ ]:
velocity = train_branch(df, combined=False, splits=splits)
combined = train_branch(df, combined=True, splits=splits)
print("Epochs:", velocity["epochs"], combined["epochs"])
results = pd.DataFrame({"Velocity only":velocity["metrics"], "Velocity + candidate tremor":combined["metrics"]})
display(results)


## 3. Initial performance on unseen simulated participants

The held-out test uses participant IDs P20–P23 (354 repetitions). MAE is the primary metric in units of remaining repetitions. A warning is issued at estimated RIR ≤ 2. Precision, recall, F1, false-positive rate, and missed warnings describe that threshold on **synthetic** labels.

![MAE comparison](assets/model_mae.svg)


In [ ]:
saved = json.loads(Path("metrics.json").read_text())
display(pd.DataFrame({"Velocity only":saved["velocity_only"],"Velocity + candidate tremor":saved["velocity_plus_candidate_tremor"]}))
print(f"Relative MAE reduction: {saved['relative_MAE_change_percent']}%")
print("15% proposal target met:", saved["relative_MAE_change_percent"] >= 15)


Relative MAE reduction: 7.7%
15% proposal target met: False


### Interpretation and next experiment

The combined model has a lower MAE in this deterministic synthetic run, but its warning recall is lower. The 15% target is not achieved. These numbers are useful for checking code, interface, and evaluation logic only. They cannot validate tremor extraction or RIR estimation in people. Next, collect approved and supervised recordings, verify video signal quality and true endpoint labels, retrain causal TCN variants using participant-grouped splits, and measure full end-to-display p95 latency.
